In [1]:
import pandas as pd 
import numpy as np 
from glob import glob
import os
import geopandas as gpd

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.fkhmqgc2M3/ipykernel_2275205/471158661.py:5: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_

In [ ]:
# 1/25 
# - need to add in coincident events (e.g. do the same timestamps have heavy snow and lake effect snow?)
# - note that ncei doesnt have snow squall. Is there a better squall observational dataset? if not this could be a contribution of my work. Ask heather AMS
# - since we have certain data prep logic for whether to run every 5 min, 30 min, etc., depending on the quantity of events (NCEI had a lot of Lake-Effect Snow, for example) - note that this logic is in the ncei_dataset_analysis.ipynb, - rather than tracking/wrting down what logic we used for what events, need to have a system where the data w/ the range of datetimes (basicaly all the data prep) is saved out somewhere, i.e. have that prepped events dataset saved out somewhere in a csv with proper naming convention, so that way 1) we dont need to 'remember/track' what logic we had to do for certain events and 2) we dont need to run that same logic again when it comes time for analysis/modeling!
# - regarding that ^ note, I am getting this set up for NCEI in the events_ofinterest dir. But did not do this for the NWS warnings dataset. Although those are easy to remember bc i never adjusted the logic, I just did the 5 min intervals, wth 15 min windows on each end of the warning,  every 5 min run, and all years (no filtering). (see /home/csutter/DRIVE-clean/NWS_warnings/notebooks/nws_dataset_analysis.ipynb for the details there). But that was simpler, I only started adjusting the data prep/ model run logic for NCEI data

# - IMPORTANT! If there are no cron logged images at all for a given datetime run instance, nothing will be saved out beyond an empty data_1_images directory. Don't have a clean way to deal with this for tracking this situation yet (since in the ncei_events_ofint directory this event will still be listed, it's just impossible for us to get model runs for it).  For an example of this situation, see /home/csutter/DRIVE-clean/operational_runs/set35_blizzard8_EgNoImgs
# - Also need to keep in mind to remove events that were used to label the data

Prepare NWS labeled dataset

['Blizzard Warning', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Snow Squall Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory']

In [2]:
# datetimes and event names (reference code in : /home/csutter/DRIVE-clean/NWS_warnings/notebooks/dataset_analysis.ipynb)
# may take ~40 seconds to read in this data

#### 1. read in data: 
d = gpd.read_file("/home/csutter/DRIVE-clean/NWS_warnings/data/nws_warnings/nws_all_warnings_cleaned.gpkg")

d.head(4)

#### 2. add timedelta duration col back (using duration_sec col)
d["duration"] = pd.to_timedelta(d["duration_sec"], unit="s")

#### 3. Subset to squalls (or whatever event type)
d_event_subset = d[d["name"]=='Snow Squall Warning'] 


#### 4. Add in range of 5-min intervals in entire duration of warning

d_event_subset["start_round"] = d_event_subset["issued"].dt.round("5min") # will want these in a df for reference of the rounded start time
d_event_subset["end_round"] = d_event_subset["expired"].dt.round("5min") # will want these in a df for reference of the rounded ende time

d_event_subset["start_buffer"] = d_event_subset["start_round"] - pd.Timedelta(minutes=15)
d_event_subset["end_buffer"] = d_event_subset["end_round"] + pd.Timedelta(minutes=15)

d_event_subset["all_times"] = d_event_subset.apply(
    lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="5T"),
    axis=1
)

d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
    lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
)

d_event_subset.head(4)


#### 5. make a list of all the datetimes we want to run
# nested loop, bc each row (squall instance) has a range of datetimes to run

nws_datetimes = []

for i in d_event_subset["all_times_format"]:
    for j in i:
        nws_datetimes.append(j)

#### 6. CRITICAL - subset to unique datetimes! Events in different regions will have overlapping times of warning. Since we run events statewide, only need the datetime once (i.e., it's not datetime|location)

nws_datetimes = np.unique(nws_datetimes)

print(len(nws_datetimes))

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

1089


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Grab list of datetimes already ran in inference


In [ ]:
# Grab list of all aggregated datetimes (I have reference code for this, /home/csutter/DRIVE-clean/operational_analysis/notebooks/summarize_dates_ran.ipynb)

In [11]:
inf_sets_ran = ["/home/csutter/DRIVE-clean/operational_runs/set0_test",
"/home/csutter/DRIVE-clean/operational_runs/set1_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set2_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set3_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set4_iceWTA_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set5_iceWTA_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set6_probSRsample_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set7_probSRsample_20250926",
"/home/csutter/DRIVE-clean/operational_runs/set9_winter22_JFMOnly_000816",
"/home/csutter/DRIVE-clean/operational_runs/set10_winter2223_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set11_winer2324_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set12_winter2425_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set13_winter22_JFMOnly_041220",
"/home/csutter/DRIVE-clean/operational_runs/set14_winter2223_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set15_winter2324_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set16_winter2425_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set17_summer_examples",
"/home/csutter/DRIVE-clean/operational_runs/set18_summer_examples_2",
"/home/csutter/DRIVE-clean/operational_runs/set19_summer_examples_3",
"/home/csutter/DRIVE-clean/operational_runs/set20_summer_examples_4",
"/home/csutter/DRIVE-clean/operational_runs/set21_summer_examples_5",
"/home/csutter/DRIVE-clean/operational_runs/set22_summer_examples_6",
"/home/csutter/DRIVE-clean/operational_runs/set23_squall1",
"/home/csutter/DRIVE-clean/operational_runs/set24_squall2",
"/home/csutter/DRIVE-clean/operational_runs/set25_squall3",
"/home/csutter/DRIVE-clean/operational_runs/set26_blizzard1",
"/home/csutter/DRIVE-clean/operational_runs/set27_blizzard2",
"/home/csutter/DRIVE-clean/operational_runs/set28_blizzard3",
"/home/csutter/DRIVE-clean/operational_runs/set29_blizzard4",
"/home/csutter/DRIVE-clean/operational_runs/set30_blizzard5",]

dirs_w_preds = [f"{i}/data_6_ensembling/*/*/*/*/*" for i in inf_sets_ran]

# print(dirs_w_preds)

pred_datetimes = []
pred_path = []
for dr in dirs_w_preds:
    # print(dr) # just for checking counts per dir
    # dr_files = [] # just for checking counts per dir
    listpredfiles = glob(dr)
    for fl in listpredfiles:
        # print(fl)
        i = fl.rfind("/")
        filedate = fl[i-13:i]
        pred_datetimes.append(filedate)
        # dr_files.append(filedate) # just for checking counts per dir

        ### Must also grab the tracker path which contains the predictions and the image path (should have all the data in it we need)
        pred_path.append(fl)
    # print(len(dr_files)) # just for checking counts per dir


print(len(pred_datetimes))
print(len(pred_path))


4924
4924


In [5]:
pred_datetimes[0:5]

['20250210_1000',
 '20250204_1000',
 '20250228_1000',
 '20250227_1000',
 '20250211_1000']

In [12]:
pred_path[0:5]

['/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/10/20250210_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/04/20250204_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/28/20250228_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/27/20250227_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/11/20250211_1000/finalpreds.csv']

Colocate nws event with existant inference obersvations (i.e. the already ran model runs, found above)
- Something to think about - How to deal with onset? Like a winter storm warning can encompass days... so maybe it's just applicable to snow squalls/other short lived events. 
- Application wise, do I need to use the entire EVENT or can I grab random 5-min timestamps within the event? And what if the squall was in the beginning but the warning just prolongs for public safety?

In [8]:
# Identify dates in both the nws_events and model_obs lists

nwsevent_with_preds = [d for d in pred_datetimes if d in nws_datetimes]
nonnwsevent_with_preds = [d for d in pred_datetimes if d not in nws_datetimes]

print(len(nwsevent_with_preds))
print(len(nonnwsevent_with_preds))

722
4195


Colocate cam data (lat / lon) with NWS event polygon

In [10]:
# need to use the raw dataframes (not just list of dates) for that
# NWS events df: d_event_subset, note that this contains all of the events which are unique by geometry|event|timeframe
# Model preds df: VARIOUS of them, need to see list of them; the list of the pred csvs are in the list called pred_path

d_event_subset.head(4)
print(type(d_event_subset))
print(d_event_subset.dtypes)

<class 'geopandas.geodataframe.GeoDataFrame'>
Unnamed: 0                    int64
vtec_year                     int64
iso_issued                   object
issued               datetime64[ns]
iso_expired                  object
expired              datetime64[ns]
eventid                       int64
phenomena                    object
significance                 object
hvtec_nwsli                  object
wfo                          object
ugc                          object
product_id                   object
name                         object
ph_name                      object
sig_name                     object
url                          object
location_type                object
ugc_gis                      object
loc_desc                     object
duration_sec                float64
geometry                   geometry
duration            timedelta64[ns]
start_round          datetime64[ns]
end_round            datetime64[ns]
start_buffer         datetime64[ns]
end_buffer        

Sample so that events and non-events are represented
- If we don't have all, can just start w snow squalls as a "proof of concept" that this idea may work...

Gather labeled dataset for model training
- cols for model inputs: img-only pred (5-cat), final pred (5-cat), nonobs pred (2-cat), raw weather data (6 vars) (rational for weather data is that, even tho this data is embedded in the final pred, the model still has errors so maybe adding in the weather data - which esp makes sense given it's nws weather events we're predicting - will help w extraneous/fine tuning. O/w, why would we expect the img-only pred or final pred to align exactly w nws events? it's different data.)
- cols for model outputs (nws event label): one hot encode or whatever

Preprocess data
- normalize, etc. 

Train model
- For now, just proof of concept w a random forest (something easy)